In [53]:
import torch
import torch.nn as nn
import random
import string


In [54]:
# CONFIG
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
D_FF = 128
MAX_LEN = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [55]:
MODEL_A_PATH = "minigpt_reverse_step2_pure_rl.pth"
MODEL_B_PATH = "minigpt_reverse_step2_Hybrid_rl.pth"


In [56]:
# VOCAB
special_tokens = ["<PAD>", "<BOS>", "<SEP>", "<EOS>"]
letters = list(string.ascii_lowercase)

itos = special_tokens + letters
stoi = {ch: i for i, ch in enumerate(itos)}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
SEP_ID = stoi["<SEP>"]
EOS_ID = stoi["<EOS>"]

VOCAB_SIZE = len(itos)

In [57]:
# MODEL
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_emb = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.pos_emb = nn.Embedding(MAX_LEN, D_MODEL)

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_FF,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            decoder_layer,
            num_layers=N_LAYERS
        )

        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, input_ids):
        seq_len = input_ids.shape[1]

        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=input_ids.device),
            diagonal=1
        ).bool()

        x = self.transformer(x, mask=causal_mask)
        logits = self.lm_head(x)

        return logits



In [58]:
# TARGET FUNCTION
def make_target(text):
    return "".join([ch for ch in reversed(text) if ch != "b"])

In [59]:
# DATA (remove b)
def random_sample(min_len, max_len, alphabet=letters):
    length = random.randint(min_len, max_len)
    x = "".join(random.choice(alphabet) for _ in range(length))
    y = make_target(x)
    return x, y

def generate_test_set(mode="easy", n_samples=1000):
    samples = []

    if mode == "easy":
        for _ in range(n_samples):
            samples.append(random_sample(2, 6))

    elif mode == "b_heavy":
        alphabet = list("abbbbccddeeff")
        for _ in range(n_samples):
            samples.append(random_sample(3, 6, alphabet))

    elif mode == "edge_cases":
        fixed_cases = [
        "bbbbb",
        "bbbabb",
        "bbccb",
        "bccbb",
        "bdbdbd",
        "cdcbdc",
        "bxxb",
        "bbzz",
        "zzbzz",
        "bbqrr",
        "qbbqr",
        "btttb",
        "ttbbtt",
        "bbnnmm",
        "mmnnbb",
        "zzbb",
        ]

        for _ in range(n_samples):
            x = random.choice(fixed_cases)
            samples.append((x, make_target(x)))

    else:
        raise ValueError("Unknown mode")

    return samples

In [60]:
# GENERATE FUNCTION
@torch.no_grad()
def generate(model, text, max_new_tokens=20):
    model.eval()

    tokens = ["<BOS>"] + list(text) + ["<SEP>"]
    ids = [stoi[t] for t in tokens]

    for _ in range(max_new_tokens):
        input_ids = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

        if input_ids.shape[1] >= MAX_LEN:
            break

        logits = model(input_ids)
        next_id = torch.argmax(logits[0, -1]).item()

        ids.append(next_id)

        if next_id == EOS_ID:
            break

    decoded = [itos[i] for i in ids]
    return decoded

def extract_prediction(output_tokens):
    if "<SEP>" in output_tokens:
        pred = output_tokens[output_tokens.index("<SEP>") + 1:]
    else:
        pred = []

    if "<EOS>" in pred:
        pred = pred[:pred.index("<EOS>")]

    pred = [t for t in pred if t not in ["<PAD>", "<BOS>", "<SEP>", "<EOS>"]]
    return "".join(pred)

In [61]:
# EVALUATE
@torch.no_grad()
def evaluate_model(model, samples):
    exact_match = 0
    total_chars = 0
    correct_chars = 0
    b_leak_count = 0
    length_error_sum = 0

    examples = []

    for x, y in samples:
        output_tokens = generate(model, x)
        pred_str = extract_prediction(output_tokens)

        if pred_str == y:
            exact_match += 1

        if "b" in pred_str:
            b_leak_count += 1

        length_error_sum += abs(len(pred_str) - len(y))

        max_len = max(len(pred_str), len(y))
        for i in range(max_len):
            p = pred_str[i] if i < len(pred_str) else None
            t = y[i] if i < len(y) else None
            if p == t:
                correct_chars += 1
            total_chars += 1

        if len(examples) < 5 and pred_str != y:
            examples.append((x, y, pred_str))

    n = len(samples)

    return {
        "exact_acc": exact_match / n,
        "char_acc": correct_chars / total_chars if total_chars > 0 else 0,
        "b_leak_rate": b_leak_count / n,
        "avg_length_error": length_error_sum / n,
        "wrong_examples": examples
    }


In [62]:
# LOAD MODEL
def load_model(path):
    model = MiniGPT().to(DEVICE)
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model


In [63]:
# MAIN
if __name__ == "__main__":
    print("Loading models")

    model_A = load_model(MODEL_A_PATH)
    model_B = load_model(MODEL_B_PATH)

    modes = [
    "easy",
    "b_heavy",
    "edge_cases"
]

    for mode in modes:
        print(f"\nTEST SET: {mode}")

        samples = generate_test_set(mode=mode, n_samples=1000)

        result_A = evaluate_model(model_A, samples)
        result_B = evaluate_model(model_B, samples)

        print("\nVersion A")
        print(f"Exact Acc       : {result_A['exact_acc']:.3f}")
        print(f"Char Acc        : {result_A['char_acc']:.3f}")
        print(f"B Leakage Rate  : {result_A['b_leak_rate']:.3f}")
        print(f"Avg Len Error   : {result_A['avg_length_error']:.3f}")

        print("\nVersion B")
        print(f"Exact Acc       : {result_B['exact_acc']:.3f}")
        print(f"Char Acc        : {result_B['char_acc']:.3f}")
        print(f"B Leakage Rate  : {result_B['b_leak_rate']:.3f}")
        print(f"Avg Len Error   : {result_B['avg_length_error']:.3f}")

        print("\nWrong examples from Version A:")
        for x, y, pred in result_A["wrong_examples"]:
            print(f"Input: {x:12s} | Expected: {y:12s} | Pred: {pred}")

        print("\nWrong examples from Version B:")
        for x, y, pred in result_B["wrong_examples"]:
            print(f"Input: {x:12s} | Expected: {y:12s} | Pred: {pred}")

Loading models

TEST SET: easy

Version A
Exact Acc       : 0.853
Char Acc        : 0.914
B Leakage Rate  : 0.053
Avg Len Error   : 0.159

Version B
Exact Acc       : 0.997
Char Acc        : 0.999
B Leakage Rate  : 0.000
Avg Len Error   : 0.003

Wrong examples from Version A:
Input: biqrb        | Expected: rqi          | Pred: rrqib
Input: qbca         | Expected: acq          | Pred: acqq
Input: awpbm        | Expected: mpwa         | Pred: mppwa
Input: tbvseo       | Expected: oesvt        | Pred: oesvtt
Input: tbwgfi       | Expected: ifgwt        | Pred: ifgwtt

Wrong examples from Version B:
Input: kkc          | Expected: ckk          | Pred: ck
Input: zzbjjg       | Expected: gjjzz        | Pred: gjjz
Input: llxpks       | Expected: skpxll       | Pred: skpxl

TEST SET: b_heavy

Version A
Exact Acc       : 0.215
Char Acc        : 0.535
B Leakage Rate  : 0.392
Avg Len Error   : 1.353

Version B
Exact Acc       : 0.970
Char Acc        : 0.986
B Leakage Rate  : 0.000
Avg Len Error